In [1]:
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import HDBSCAN
from stopwordsiso import stopwords
from tqdm import tqdm
import ast
from tqdm.contrib.concurrent import process_map

import warnings
warnings.simplefilter("ignore")

/hpc/home/as1676/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../../../data/comp_ideology_detection/interventions_us_113_with_embeddings.csv', index_col=0)

In [3]:
df['word_count'] = df['speech'].str.split().str.len()
df = df[df['word_count'] <= 600]
print(f"nrows: {df.shape[0]}")

tqdm.pandas()
df['embedding'] = df['embedding'].progress_apply(ast.literal_eval)

docs = df['speech'].tolist()
embeddings = np.vstack(df['embedding'].values)

nrows: 67802


100%|██████████| 67802/67802 [02:49<00:00, 401.15it/s]


In [4]:
custom_stopwords = [
    'mr', 'mrs', 'madam', 'speaker', 'president', 'gentleman', 'gentlewoman',
    'yield', 'time', 'today', 'would', 'like', 'also', 'one', 'us', 'say',
    'want', 'know', 'think', 'thank', 'colleague', 'member', 'congress',
    'house', 'senate', 'legislation', 'bill', 'act', 'ask', 'unanimous',
    'consent', 'rise'
]
all_stopwords = list(set(list(stopwords('en')) + custom_stopwords))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    min_df=10,          
    max_df=0.5,       
    ngram_range=(1, 2)
)

In [5]:
umap_model = UMAP(random_state=123, n_jobs=18)
hdbscan_model = HDBSCAN(min_cluster_size=100, n_jobs=18)

topic_model = BERTopic(verbose=True,
                      umap_model=umap_model,
                      hdbscan_model=hdbscan_model,
                      vectorizer_model=CountVectorizer(stop_words=list(stopwords('en'))))

topics, probs = topic_model.fit_transform(docs, embeddings)

2026-02-16 13:44:56,283 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-16 13:46:36,695 - BERTopic - Dimensionality - Completed ✓
2026-02-16 13:46:36,696 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-16 13:47:03,567 - BERTopic - Cluster - Completed ✓
2026-02-16 13:47:03,580 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-16 13:47:07,777 - BERTopic - Representation - Completed ✓


In [17]:
topics_reduced = topic_model.reduce_outliers(
    documents = docs, 
    topics = topics, 
    embeddings = embeddings,
    strategy="embeddings" 
)

In [21]:
topic_model.update_topics(docs, topics=topics_reduced)

2026-02-16 14:16:11,356 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [22]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,5834,0_yield_gentleman_minutes_from,"[yield, gentleman, minutes, from, speaker, min...",[Mr. Speaker. I yield 2 minutes to the gentlem...
1,1,3835,1_his_he_and_was,"[his, he, and, was, in, of, the, as, to, service]",[Mr. President. in Arkansas. our law enforceme...
2,2,2250,2_budget_we_the_to,"[budget, we, the, to, that, our, is, it, and, ...",[I thank the gentleman for his leadership. and...
3,3,1810,3_care_health_insurance_obamacare,"[care, health, insurance, obamacare, affordabl...",[Madam Speaker. ever since President Obama was...
4,4,1579,4_energy_oil_gas_is,"[energy, oil, gas, is, the, pipeline, of, that...",[Mr. Speaker. again and again we have heard fr...
...,...,...,...,...,...
72,72,307,72_request_unanimous_consent_purpose,"[request, unanimous, consent, purpose, yield, ...",[Mr. Speaker. for the purpose of a unanimous c...
73,73,1009,73_amendment_this_the_to,"[amendment, this, the, to, that, in, is, of, i...",[Madam Chair. I rise in support of the amendme...
74,74,265,74_adjourn_move_now_house,"[adjourn, move, now, house, do, speaker, mr, t...",[Mr. Speaker. I move that the House do now adj...
75,75,285,75_absence_suggest_quorum_president,"[absence, suggest, quorum, president, mr, mada...",[Mr. President. I suggest the absence of a quo...


In [25]:
df['topic'] = topics_reduced

df.to_csv('../../../data/comp_ideology_detection/interventions_us_113_with_embeddings.csv')
topic_model.get_topic_info().to_csv('../../../data/comp_ideology_detection/topic_info.csv')